# 02 — Procesamiento Distribuido con Dask (Distributed — Dataproc)

Versión distribuida real. Usa el **Dask Scheduler** en el master y los **Dask Workers** en los nodos worker.
Cada worker lee desde GCS, procesa y devuelve solo el **resultado agregado** — los datos crudos nunca llegan al master.

**Flujo:** GCS raw/ (2017–2025) → `client.map` (workers) → agregados pequeños → master combina → BigQuery

| # | Operación | Descripción | Tabla BigQuery |
|---|-----------|-------------|----------------|
| 1 | **Create** | Distribución horaria de crímenes por era COVID | `hourly_distribution` |
| 2 | **Read**   | Top 10 tipos de crimen por año (2017–2025) | — (exploración) |
| 3 | **Read**   | Tasa de arresto por tipo de crimen y era | `arrest_rate_by_type` |
| 4 | **Update** | Añadir `severity_category` por código FBI | `severity_distribution` |
| 5 | **Delete** | Eliminar registros sin coordenadas geográficas | — (limpieza) |

In [1]:
from dask.distributed import Client
import pandas as pd
import time
import os

PROJECT_ID = 'my-first-project-492901'
DATASET_ID = 'chicago_crimes_results'
YEARS_ANALYSIS = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
ERA_MAP = {2017:'PRE', 2018:'PRE', 2019:'PRE',
           2020:'DURANTE', 2021:'DURANTE', 2022:'DURANTE',
           2023:'POST', 2024:'POST', 2025:'POST'}

def to_bigquery(df, table_name):
    df.to_gbq(destination_table=f'{DATASET_ID}.{table_name}',
              project_id=PROJECT_ID, if_exists='replace', progress_bar=False)
    print(f'  → BigQuery: {DATASET_ID}.{table_name}  ({len(df):,} filas)')

client = Client('localhost:8786')
print(client)
print(f'Workers activos: {len(client.scheduler_info()["workers"])}')
for wid, w in client.scheduler_info()['workers'].items():
    print(f'  {wid}  ncores={w["nthreads"]}')

<Client: 'tcp://10.128.0.56:8786' processes=2 threads=8, memory=22.35 GiB>
Workers activos: 2
  tcp://10.128.0.55:42759  ncores=4
  tcp://10.128.0.57:43975  ncores=4


In [2]:
# ── Función que corre completamente en los workers ────────────────────────────
# Cada worker: lee 1 año desde GCS → calcula TODOS los agregados → devuelve solo resultados pequeños
def process_year_all_cruds(year):
    import pandas as pd
    import gcsfs

    ERA = {2017:'PRE', 2018:'PRE', 2019:'PRE',
           2020:'DURANTE', 2021:'DURANTE', 2022:'DURANTE',
           2023:'POST', 2024:'POST', 2025:'POST'}
    VIOLENT_FBI = {'01A','01B','02','03','04A','04B','05','06','07','09'}
    MEDIUM_FBI  = {'08A','08B','10','11','12'}

    # Leer desde GCS directamente en el worker
    fs = gcsfs.GCSFileSystem(project='my-first-project-492901', token='google_default')
    with fs.open(f'big-data-proyecto-parcial/raw/Chicago_Crimes_{year}.csv') as f:
        df = pd.read_csv(f, dtype={
            'unique_key':'Int64','beat':'Int64','district':'Int64',
            'ward':'Int64','community_area':'Int64',
            'x_coordinate':'Int64','y_coordinate':'Int64','year':'Int64',
            'latitude':'float64','longitude':'float64',
        }, parse_dates=['date'], on_bad_lines='skip')

    df['covid_era']   = df['year'].map(ERA)
    df['hour_of_day'] = df['date'].dt.hour
    df['severity']    = df['fbi_code'].apply(
        lambda c: 'HIGH' if c in VIOLENT_FBI else ('MEDIUM' if c in MEDIUM_FBI else 'LOW'))

    n_total = len(df)

    # CRUD 1 — hourly aggregation
    hourly = df.groupby(['hour_of_day','covid_era']).agg(
        total_crimes=('unique_key','count'), arrests=('arrest','sum')).reset_index()

    # CRUD 2 — top types
    top_types = df.groupby(['year','primary_type']).size().reset_index(name='count')

    # CRUD 3 — arrest rate by type and era
    arrest_era = df.groupby(['primary_type','covid_era']).agg(
        arrest_sum=('arrest','sum'), total=('unique_key','count')).reset_index()

    # CRUD 4 — severity distribution
    severity = df.groupby(['covid_era','severity']).size().reset_index(name='count')

    # CRUD 5 — missing coordinates
    n_missing = int(df['latitude'].isna().sum())
    n_clean   = int(df['latitude'].notna().sum())

    # Solo devuelve agregados (KB), no el DataFrame completo (GB)
    return {
        'year': year, 'n_total': n_total,
        'n_missing_coord': n_missing, 'n_clean_coord': n_clean,
        'hourly': hourly, 'top_types': top_types,
        'arrest_era': arrest_era, 'severity': severity,
    }

# Distribuir los 9 años a los workers en paralelo
t0 = time.time()
print('Distribuyendo 9 años a los workers...')
futures = client.map(process_year_all_cruds, YEARS_ANALYSIS)
results = client.gather(futures)   # solo trae agregados pequeños al master
elapsed = time.time() - t0

total = sum(r['n_total'] for r in results)
print(f'Procesamiento distribuido completado en {elapsed:.1f}s')
print(f'Registros procesados: {total:,}')
print(f'Workers activos:      {len(client.scheduler_info()["workers"])}')
for wid, w in client.scheduler_info()['workers'].items():
    print(f'  {wid}  ncores={w["nthreads"]}')

Distribuyendo 9 años a los workers...
Procesamiento distribuido completado en 14.5s
Registros procesados: 2,072,943
Workers activos:      2
  tcp://10.128.0.55:42759  ncores=4
  tcp://10.128.0.57:43975  ncores=4


---
## CRUD 1 — CREATE: Distribución horaria por era COVID

In [3]:
hourly = (
    pd.concat([r['hourly'] for r in results], ignore_index=True)
    .groupby(['hour_of_day','covid_era'])
    .agg(total_crimes=('total_crimes','sum'), arrests=('arrests','sum'))
    .reset_index()
    .sort_values(['hour_of_day','covid_era'])
)
hourly['arrest_rate_pct'] = (hourly['arrests'] / hourly['total_crimes'] * 100).round(2)

night = hourly[(hourly['hour_of_day'] >= 22) | (hourly['hour_of_day'] <= 5)]
night_era = night.groupby('covid_era')['total_crimes'].sum()
total_era  = hourly.groupby('covid_era')['total_crimes'].sum()
print('Índice de criminalidad nocturna por era:')
for era in ['PRE','DURANTE','POST']:
    idx = night_era.get(era, 0) / total_era.get(era, 1) * 100
    print(f'  {era:<8}: {idx:.2f}%')

print(f'Workers activos: {len(client.scheduler_info()["workers"])}')
to_bigquery(hourly[['hour_of_day','covid_era','total_crimes','arrests','arrest_rate_pct']], 'hourly_distribution')

Índice de criminalidad nocturna por era:
  PRE     : 24.42%
  DURANTE : 28.01%
  POST    : 27.99%
Workers activos: 2
  → BigQuery: chicago_crimes_results.hourly_distribution  (72 filas)


---
## CRUD 2 — READ: Top 10 tipos de crimen por año

In [4]:
top_by_year = (
    pd.concat([r['top_types'] for r in results], ignore_index=True)
    .groupby(['year','primary_type'])['count'].sum()
    .reset_index()
    .sort_values(['year','count'], ascending=[True,False])
)
top10 = top_by_year.groupby('year').head(10).reset_index(drop=True)

print('Top 5 crímenes por año:')
ERA_MAP = {2017:'PRE', 2018:'PRE', 2019:'PRE',
           2020:'DURANTE', 2021:'DURANTE', 2022:'DURANTE',
           2023:'POST', 2024:'POST', 2025:'POST'}
for y in YEARS_ANALYSIS:
    era = ERA_MAP[y]
    top5 = top10[top10['year'] == y].head(5)
    types_str = ', '.join(f"{r['primary_type']}({r['count']:,})" for _, r in top5.iterrows())
    print(f'  {y} [{era}]: {types_str}')

Top 5 crímenes por año:
  2017 [PRE]: THEFT(64,386), BATTERY(49,239), CRIMINAL DAMAGE(29,045), DECEPTIVE PRACTICE(19,742), ASSAULT(19,306)
  2018 [PRE]: THEFT(65,290), BATTERY(49,832), CRIMINAL DAMAGE(27,823), ASSAULT(20,407), DECEPTIVE PRACTICE(19,927)
  2019 [PRE]: THEFT(62,497), BATTERY(49,522), CRIMINAL DAMAGE(26,682), ASSAULT(20,623), DECEPTIVE PRACTICE(19,190)
  2020 [DURANTE]: BATTERY(41,515), THEFT(41,344), CRIMINAL DAMAGE(24,878), DECEPTIVE PRACTICE(18,519), ASSAULT(18,258)
  2021 [DURANTE]: THEFT(40,819), BATTERY(40,472), CRIMINAL DAMAGE(25,096), ASSAULT(20,343), DECEPTIVE PRACTICE(17,770)
  2022 [DURANTE]: THEFT(54,890), BATTERY(40,949), CRIMINAL DAMAGE(27,248), MOTOR VEHICLE THEFT(21,466), ASSAULT(20,809)
  2023 [POST]: THEFT(57,469), BATTERY(44,223), CRIMINAL DAMAGE(30,087), MOTOR VEHICLE THEFT(29,251), ASSAULT(22,626)
  2024 [POST]: THEFT(59,956), BATTERY(46,004), CRIMINAL DAMAGE(28,497), ASSAULT(23,400), MOTOR VEHICLE THEFT(21,633)
  2025 [POST]: THEFT(21,054), BATTERY(1

---
## CRUD 3 — READ: Tasa de arresto por tipo de crimen y era COVID

In [5]:
arrest_df = (
    pd.concat([r['arrest_era'] for r in results], ignore_index=True)
    .groupby(['primary_type','covid_era'])
    .agg(arrest_sum=('arrest_sum','sum'), total_crimes=('total','sum'))
    .reset_index()
)
arrest_df['arrest_rate_pct'] = (arrest_df['arrest_sum'] / arrest_df['total_crimes'] * 100).round(2)
result3 = arrest_df[['primary_type','covid_era','total_crimes','arrest_rate_pct']].sort_values(['primary_type','covid_era'])

print('Variación de tasa de arresto (top 8 tipos más frecuentes):')
top_types_idx = result3.groupby('primary_type')['total_crimes'].sum().nlargest(8).index
pivot = result3[result3['primary_type'].isin(top_types_idx)].pivot(
    index='primary_type', columns='covid_era', values='arrest_rate_pct')
print(pivot[['PRE','DURANTE','POST']].to_string())

print(f'Workers activos: {len(client.scheduler_info()["workers"])}')
to_bigquery(result3, 'arrest_rate_by_type')

Variación de tasa de arresto (top 8 tipos más frecuentes):
covid_era              PRE  DURANTE   POST
primary_type                              
ASSAULT              17.52    10.38  10.56
BATTERY              20.72    15.17  16.24
CRIMINAL DAMAGE       6.11     3.83   3.52
DECEPTIVE PRACTICE    4.49     1.71   3.64
MOTOR VEHICLE THEFT   6.55     3.44   2.76
OTHER OFFENSE        21.55     13.0  18.36
ROBBERY               8.24     6.42   5.63
THEFT                 9.85     4.66   6.43
Workers activos: 2
  → BigQuery: chicago_crimes_results.arrest_rate_by_type  (95 filas)


---
## CRUD 4 — UPDATE: Añadir `severity_category`

In [6]:
severity_df = (
    pd.concat([r['severity'] for r in results], ignore_index=True)
    .groupby(['covid_era','severity'])['count'].sum()
    .reset_index()
)
era_totals = severity_df.groupby('covid_era')['count'].transform('sum')
severity_df['percentage'] = (severity_df['count'] / era_totals * 100).round(2)

print('Distribución de severidad por era COVID:')
pivot_sev = severity_df.pivot(index='severity', columns='covid_era', values='percentage')
print(pivot_sev[['PRE','DURANTE','POST']].to_string())

to_bigquery(severity_df, 'severity_distribution')

Distribución de severidad por era COVID:
covid_era    PRE  DURANTE   POST
severity                        
HIGH       15.19    17.71  20.06
LOW        61.43    57.55  55.28
MEDIUM     23.38    24.74  24.66
  → BigQuery: chicago_crimes_results.severity_distribution  (9 filas)


---
## CRUD 5 — DELETE: Eliminar registros sin coordenadas geográficas

In [7]:
print('Registros sin coordenadas por año y era COVID:')
for r in results:
    era = ERA_MAP[r['year']]
    pct = r['n_missing_coord'] / r['n_total'] * 100 if r['n_total'] else 0
    print(f'  {r["year"]} [{era}]: total={r["n_total"]:,}  sin_coord={r["n_missing_coord"]:,}  ({pct:.2f}%)')

n_clean = sum(r['n_clean_coord'] for r in results)
n_total = sum(r['n_total'] for r in results)
print(f'\nRegistros originales:           {n_total:,}')
print(f'Registros con coordenadas:      {n_clean:,}')
print(f'Registros eliminados (DELETE):  {n_total - n_clean:,}')
print(f'Workers utilizados: {len(client.scheduler_info()["workers"])}')

client.close()
print('\nCliente Dask cerrado. Resultados guardados en BigQuery.')

Registros sin coordenadas por año y era COVID:
  2017 [PRE]: total=269,214  sin_coord=4,246  (1.58%)
  2018 [PRE]: total=269,070  sin_coord=5,514  (2.05%)
  2019 [PRE]: total=261,555  sin_coord=2,383  (0.91%)
  2020 [DURANTE]: total=212,522  sin_coord=4,567  (2.15%)
  2021 [DURANTE]: total=209,406  sin_coord=6,538  (3.12%)
  2022 [DURANTE]: total=239,655  sin_coord=4,771  (1.99%)
  2023 [POST]: total=262,756  sin_coord=1,499  (0.57%)
  2024 [POST]: total=256,305  sin_coord=180  (0.07%)
  2025 [POST]: total=92,460  sin_coord=711  (0.77%)

Registros originales:           2,072,943
Registros con coordenadas:      2,042,534
Registros eliminados (DELETE):  30,409
Workers utilizados: 2

Cliente Dask cerrado. Resultados guardados en BigQuery.
